<a href="https://colab.research.google.com/github/heyitsraditya/akomodasi_kintamani/blob/main/akomodasi_kintamani_2501010024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import importlib.util
import subprocess
import sys

for paket in ("reportlab", "geopandas", "shapely", "pandas", "requests", "matplotlib"):
    if importlib.util.find_spec(paket) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paket])
        importlib.invalidate_caches()

import hashlib
import json
import os
import re
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path
from xml.sax.saxutils import escape

import matplotlib
import pandas as pd
import requests
import geopandas as gpd
from shapely.geometry import LineString
from shapely.ops import polygonize, unary_union

matplotlib.use("Agg")
import matplotlib.pyplot as plt

NAMA = "I Gede Raditya Ananda Putra"
NIM = "2501010024"
URL = os.environ.get("OVERPASS_URL", "https://overpass-api.de/api/interpreter")
HEADERS = {"User-Agent": f"INSTIKI-BIFP243-Praktikum/1.0 ({NIM}@instiki.ac.id)"}
BBOX = (-8.35, 115.27, -8.12, 115.49)
RELATION_ID = 20447233
FILTER = '["tourism"~"^(hotel|guest_house|hostel|motel|apartment|chalet)$"]'

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, INTERIM, DOCS, REPORTS = (ROOT / p for p in ("data/raw", "data/interim", "docs", "reports"))
for folder in (RAW, INTERIM, DOCS, REPORTS):
    folder.mkdir(parents=True, exist_ok=True)

QUERY_AREA = f"""[out:json][timeout:90];
area["name"="Kintamani"]["boundary"="administrative"]["admin_level"="6"]->.kec;
nwr{FILTER}(area.kec);
out center tags;"""
QUERY_BBOX = f"""[out:json][timeout:90];
nwr{FILTER}({BBOX[0]},{BBOX[1]},{BBOX[2]},{BBOX[3]});
out center tags;"""

QUERY_BATAS = f"""[out:json][timeout:90];
relation({RELATION_ID});
out geom;"""

print("Folder proyek:", ROOT)
print("Endpoint:", URL)
print("Pengambil:", NAMA, NIM)

Folder proyek: /content
Endpoint: https://overpass-api.de/api/interpreter
Pengambil: I Gede Raditya Ananda Putra 2501010024


In [ ]:
def md5_berkas(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for blok in iter(lambda: f.read(1 << 20), b""):
            h.update(blok)
    return h.hexdigest()


def ambil_overpass(query, log, maks_percobaan=4, jeda=5):
    for percobaan in range(1, maks_percobaan + 1):
        try:
            r = requests.post(URL, data={"data": query}, headers=HEADERS, timeout=120)
            if r.status_code == 200:
                data = r.json()
                if "runtime error" not in str(data.get("remark", "")).lower():
                    log.append(f"percobaan {percobaan}: status 200")
                    print(log[-1])
                    return data
                log.append(f"percobaan {percobaan}: status 200 dengan remark runtime error")
            else:
                log.append(f"percobaan {percobaan}: status {r.status_code}")
        except (requests.Timeout, requests.ConnectionError, ValueError) as e:
            log.append(f"percobaan {percobaan}: {type(e).__name__}")
        print(log[-1])
        if percobaan < maks_percobaan:
            print(f"tunggu {jeda}s")
            time.sleep(jeda)
            jeda *= 2
    raise RuntimeError("Gagal mengambil data dari Overpass")


def simpan_mentah(data, awalan, stamp):
    path = RAW / f"{awalan}_{stamp}.json"
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    path.chmod(0o444)
    return path


def ambil_batas(log):
    try:
        batas = ambil_overpass(QUERY_BATAS, log)
        tg = batas["elements"][0].get("tags", {})
        if tg.get("name") != "Kintamani" or tg.get("admin_level") != "6":
            raise ValueError("relasi bukan Kecamatan Kintamani admin_level 6")
        return batas
    except (RuntimeError, ValueError, KeyError, IndexError) as e:
        log.append(f"batas administratif gagal diambil: {type(e).__name__}")
        print(log[-1])
        return None


def akuisisi():
    prov_path = DOCS / "provenance.json"
    if prov_path.exists() and not os.environ.get("FORCE_FETCH"):
        prov = json.loads(prov_path.read_text(encoding="utf-8"))
        raw_path = RAW / prov["berkas_mentah"]["nama"]
        utuh = raw_path.exists() and md5_berkas(raw_path) == prov["berkas_mentah"]["md5"]
        batas = None
        info = prov.get("berkas_batas")
        if info:
            bp = RAW / info["nama"]
            utuh = utuh and bp.exists() and md5_berkas(bp) == info["md5"]
            if utuh:
                batas = json.loads(bp.read_text(encoding="utf-8"))
        if utuh:
            return json.loads(raw_path.read_text(encoding="utf-8")), batas, prov, raw_path, False
    log = []
    for metode, query in (("area_administratif", QUERY_AREA), ("bounding_box", QUERY_BBOX)):
        data = ambil_overpass(query, log)
        if data.get("elements"):
            break
        log.append(f"metode {metode}: 0 elemen, beralih ke metode berikutnya")
        print(log[-1])
    else:
        raise RuntimeError("Query tidak mengembalikan elemen")
    batas = ambil_batas(log)
    waktu = datetime.now(timezone.utc)
    stamp = waktu.strftime("%Y%m%d_%H%M%S")
    raw_path = simpan_mentah(data, "osm_akomodasi_kintamani", stamp)
    berkas_batas = None
    if batas:
        bp = simpan_mentah(batas, "osm_batas_kintamani", stamp)
        berkas_batas = {"nama": bp.name, "md5": md5_berkas(bp), "relation_id": RELATION_ID, "query": QUERY_BATAS}
    (DOCS / "query_overpass.txt").write_text(query + "\n", encoding="utf-8")
    prov = {
        "url_endpoint": URL,
        "metode_wilayah": metode,
        "query": query,
        "waktu_akuisisi_utc": waktu.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "jumlah_elemen": len(data["elements"]),
        "berkas_mentah": {"nama": raw_path.name, "md5": md5_berkas(raw_path)},
        "berkas_batas": berkas_batas,
        "lisensi": "ODbL 1.0 - (c) OpenStreetMap contributors",
        "pengambil": {"nama": NAMA, "nim": NIM},
        "log_percobaan": log,
    }
    prov_path.write_text(json.dumps(prov, ensure_ascii=False, indent=2), encoding="utf-8")
    return data, batas, prov, raw_path, True


def bangun_poligon(batas):
    if not batas:
        return None
    garis = {"outer": [], "inner": []}
    for m in batas["elements"][0].get("members", []):
        if m.get("type") == "way" and m.get("geometry") and m.get("role") in garis:
            garis[m["role"]].append(LineString([(p["lon"], p["lat"]) for p in m["geometry"]]))
    luar = list(polygonize(unary_union(garis["outer"]))) if garis["outer"] else []
    if not luar:
        return None
    poligon = unary_union(luar)
    if garis["inner"]:
        lubang = list(polygonize(unary_union(garis["inner"])))
        if lubang:
            poligon = poligon.difference(unary_union(lubang))
    if not poligon.is_valid:
        poligon = poligon.buffer(0)
    return poligon


data, batas, prov, raw_path, baru = akuisisi()
poligon = bangun_poligon(batas)
print("Akuisisi baru:" if baru else "Memakai respons mentah tersimpan:", raw_path.name)
print("Metode wilayah:", prov["metode_wilayah"])
print("Waktu akuisisi (UTC):", prov["waktu_akuisisi_utc"])
print("Jumlah elemen:", len(data["elements"]))
print("MD5:", prov["berkas_mentah"]["md5"])
print("Poligon relasi", RELATION_ID, "terbentuk:" , poligon is not None, "| valid:", poligon.is_valid if poligon is not None else "-")

percobaan 1: status 502
tunggu 5s
percobaan 2: status 502
tunggu 10s
percobaan 3: status 200
percobaan 1: status 200
Akuisisi baru: osm_akomodasi_kintamani_20261111_121215.json
Metode wilayah: area_administratif
Waktu akuisisi (UTC): 2026-11-11T12:12:15Z
Jumlah elemen: 32
MD5: c5865055f41c1f3fcd01f75047ed2449
Poligon relasi 20447233 terbentuk: True | valid: True


In [ ]:
KOLOM = ["osm_type", "osm_id", "nama", "jenis", "lat", "lon", "desa", "bintang", "jumlah_kamar", "website", "url_osm"]
SOSMED = re.compile(r"facebook\.com|fb\.com|instagram\.com|twitter\.com|x\.com|tiktok\.com|wa\.me|whatsapp|t\.me|linktr\.ee|linkedin\.com", re.I)


def ratakan(elements):
    baris = []
    for e in elements:
        tg = e.get("tags", {})
        pusat = e.get("center", e)
        baris.append({
            "osm_type": e["type"], "osm_id": e["id"],
            "nama": tg.get("name"), "jenis": tg.get("tourism"),
            "lat": pusat.get("lat"), "lon": pusat.get("lon"),
            "desa": tg.get("addr:village"),
            "bintang": tg.get("stars"), "jumlah_kamar": tg.get("rooms"),
            "website": tg.get("website") or tg.get("contact:website"),
            "url_osm": f"https://www.openstreetmap.org/{e['type']}/{e['id']}",
        })
    return pd.DataFrame(baris, columns=KOLOM)


mentah = ratakan(data["elements"])
catatan = [("jumlah elemen awal dari respons mentah", len(mentah), "tidak ada tindakan")]

dup = mentah.duplicated(["osm_type", "osm_id"])
catatan.append(("duplikat osm_type + osm_id", int(dup.sum()), "baris pertama dipertahankan, duplikat dihapus" if dup.any() else "tidak ada tindakan"))
tabel = mentah[~dup].copy()

for k in ("lat", "lon"):
    tabel[k] = pd.to_numeric(tabel[k], errors="coerce")
tanpa_koordinat = tabel[["lat", "lon"]].isna().any(axis=1)
catatan.append(("koordinat kosong", int(tanpa_koordinat.sum()), "dihapus karena tidak dapat dipetakan" if tanpa_koordinat.any() else "tidak ada tindakan"))
tabel = tabel[~tanpa_koordinat]

if poligon is not None:
    titik = gpd.GeoSeries(gpd.points_from_xy(tabel["lon"], tabel["lat"]), index=tabel.index, crs="EPSG:4326")
    di_dalam = titik.intersects(poligon)
    metode_filter = f"poligon relasi OSM {RELATION_ID}"
else:
    di_dalam = tabel["lat"].between(BBOX[0], BBOX[2]) & tabel["lon"].between(BBOX[1], BBOX[3])
    metode_filter = "bounding box (poligon tidak tersedia)"
catatan.append((f"titik di luar wilayah Kintamani, diuji dengan {metode_filter}", int((~di_dalam).sum()), "dikeluarkan dari dataset dan disimpan di docs/data_luar_wilayah.csv" if (~di_dalam).any() else "tidak ada tindakan"))
tabel[~di_dalam].to_csv(DOCS / "data_luar_wilayah.csv", index=False, encoding="utf-8")
tabel = tabel[di_dalam]

sosmed = tabel["website"].fillna("").str.contains(SOSMED) | tabel["website"].fillna("").str.contains("@")
catatan.append(("kolom website berisi akun media sosial atau kontak pribadi", int(sosmed.sum()), "nilai dikosongkan agar tidak ada data kontak" if sosmed.any() else "tidak ada tindakan"))
tabel.loc[sosmed, "website"] = None

tanpa_nama = int(tabel["nama"].isna().sum())
catatan.append(("akomodasi tanpa nama", tanpa_nama, "dipertahankan sebagai akomodasi tak bernama; dicatat sebagai keterbatasan kualitas"))

for k in ("desa", "bintang", "jumlah_kamar", "website"):
    persen = round(tabel[k].isna().mean() * 100, 1) if len(tabel) else 100.0
    keputusan = "kolom dipertahankan; analisis berbasis kolom ini bias" if persen >= 50 else "tidak ada tindakan"
    catatan.append((f"kolom {k} kosong", f"{persen}%", keputusan))

if tabel.empty:
    raise RuntimeError("Dataset kosong setelah pemeriksaan dasar")

tabel = tabel.sort_values(["osm_type", "osm_id"]).reset_index(drop=True)
tabel.to_csv(INTERIM / "akomodasi_kintamani.csv", index=False, encoding="utf-8")
pd.DataFrame(catatan, columns=["pemeriksaan", "temuan", "keputusan"]).to_csv(DOCS / "catatan_kualitas.csv", index=False, encoding="utf-8")

print("Baris akhir:", len(tabel))
print(tabel["jenis"].value_counts())
print("Tanpa nama:", tanpa_nama)
print(pd.DataFrame(catatan, columns=["pemeriksaan", "temuan", "keputusan"]).to_string(index=False))
print(tabel.head())

Baris akhir: 32
jenis
hotel          16
guest_house     9
hostel          4
chalet          2
motel           1
Name: count, dtype: int64
Tanpa nama: 1
                                                              pemeriksaan temuan                                                                          keputusan
                                   jumlah elemen awal dari respons mentah     32                                                                 tidak ada tindakan
                                               duplikat osm_type + osm_id      0                                                                 tidak ada tindakan
                                                         koordinat kosong      0                                                                 tidak ada tindakan
titik di luar wilayah Kintamani, diuji dengan poligon relasi OSM 20447233      0                                                                 tidak ada tindakan
               kolom website

In [ ]:
per_jenis = tabel["jenis"].value_counts().rename_axis("jenis").reset_index(name="jumlah")
per_desa = tabel["desa"].fillna("(desa tidak tercatat)").value_counts().rename_axis("desa").reset_index(name="jumlah")
per_jenis.to_csv(DOCS / "ringkasan_jenis.csv", index=False, encoding="utf-8")
per_desa.to_csv(DOCS / "ringkasan_desa.csv", index=False, encoding="utf-8")

fig, ax = plt.subplots(figsize=(7, 6))
if poligon is not None:
    gpd.GeoSeries([poligon], crs="EPSG:4326").boundary.plot(ax=ax, color="black", linewidth=1.2)
for jenis, grup in tabel.groupby("jenis"):
    ax.scatter(grup["lon"], grup["lat"], s=16, alpha=0.8, label=f"{jenis} ({len(grup)})")
ax.set_xlabel("Bujur")
ax.set_ylabel("Lintang")
ax.set_title("Sebaran akomodasi di Kecamatan Kintamani")
ax.set_aspect("equal")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.text(0.99, 0.01, "© OpenStreetMap contributors", ha="right", fontsize=7)
fig.tight_layout()
fig.savefig(REPORTS / "peta_sebaran.png", dpi=150)
plt.close(fig)

print(per_jenis.to_string(index=False))
print(per_desa.head(15).to_string(index=False))
print("Peta tersimpan:", REPORTS / "peta_sebaran.png")

      jenis  jumlah
      hotel      16
guest_house       9
     hostel       4
     chalet       2
      motel       1
                 desa  jumlah
(desa tidak tercatat)      32
Peta tersimpan: /content/reports/peta_sebaran.png


In [ ]:
ketentuan_kolom = [
    ("osm_type", "teks", "Jenis objek OSM: node, way, atau relation", "way", "Tidak"),
    ("osm_id", "bilangan bulat", "ID objek di OpenStreetMap", "123456789", "Tidak"),
    ("nama", "teks", "Nama akomodasi sesuai tag name", "(nama akomodasi)", "Ya"),
    ("jenis", "teks", "Nilai tag tourism", "guest_house", "Tidak"),
    ("lat", "desimal", "Lintang titik atau titik tengah bangunan (WGS84)", "-8.27", "Tidak"),
    ("lon", "desimal", "Bujur titik atau titik tengah bangunan (WGS84)", "115.37", "Tidak"),
    ("desa", "teks", "Desa sesuai tag addr:village", "(nama desa)", "Ya"),
    ("bintang", "teks", "Klasifikasi bintang (tag stars)", "3", "Ya"),
    ("jumlah_kamar", "teks", "Jumlah kamar (tag rooms)", "20", "Ya"),
    ("website", "teks", "Situs resmi akomodasi; akun media sosial dikosongkan", "https://...", "Ya"),
    ("url_osm", "teks", "Tautan ke objek di openstreetmap.org untuk pengecekan", "https://www.openstreetmap.org/way/...", "Tidak"),
]
kamus = []
for kolom, tipe, deskripsi, contoh, kosong in ketentuan_kolom:
    nilai = tabel[kolom].dropna()
    contoh_nyata = str(nilai.iloc[0]) if len(nilai) else contoh
    kamus.append((kolom, tipe, deskripsi, contoh_nyata, kosong, round(tabel[kolom].notna().mean() * 100, 1)))
pd.DataFrame(kamus, columns=["kolom", "tipe", "deskripsi", "contoh", "boleh_kosong", "persen_terisi"]).to_csv(DOCS / "data_dictionary.csv", index=False, encoding="utf-8")

KONTAK = re.compile(r"phone|email|mail|telp|fax|whatsapp|facebook|instagram|twitter|tiktok|contact", re.I)
tanpa_kolom_kontak = not any(KONTAK.search(c) for c in tabel.columns)
tanpa_nilai_kontak = not (tabel["website"].fillna("").str.contains(SOSMED).any() or tabel["website"].fillna("").str.contains("@").any())
raw_utuh = md5_berkas(raw_path) == prov["berkas_mentah"]["md5"]

etika = [
    ("Sumber data hanya yang diizinkan", "Ya", f"Satu-satunya endpoint: {prov['url_endpoint']} (OpenStreetMap via Overpass API)"),
    ("Tidak ada scraping OTA, platform pemesanan, atau Google Maps", "Ya", "Tidak ada request ke domain selain endpoint Overpass"),
    ("Lisensi ODbL dan atribusi dicantumkan", "Ya", "Atribusi '© OpenStreetMap contributors' ada di README dan laporan"),
    ("Wilayah divalidasi dengan batas administratif", "Ya" if poligon is not None else "Tidak", f"Titik disaring terhadap poligon relasi OSM {RELATION_ID}" if poligon is not None else "Poligon tidak tersedia; memakai bounding box"),
    ("Data kontak pribadi tidak diambil", "Ya" if tanpa_kolom_kontak and tanpa_nilai_kontak else "Tidak", "Tidak ada kolom telepon, email, atau media sosial; nilai website berisi akun media sosial dikosongkan"),
    ("Minimisasi data", "Ya", f"Dataset hanya {len(KOLOM)} kolom yang diperlukan untuk inventarisasi akomodasi"),
    ("Batas wajar penggunaan Overpass dipatuhi", "Ya", f"{len(prov['log_percobaan'])} catatan percobaan; User-Agent teridentifikasi; tanpa loop request"),
    ("Respons mentah tidak diubah", "Ya" if raw_utuh else "Tidak", f"MD5 {prov['berkas_mentah']['md5']} cocok dengan berkas data/raw"),
    ("Keterbatasan data dilaporkan", "Ya", "Catatan kualitas dan refleksi kelengkapan tercantum di laporan"),
]
pd.DataFrame(etika, columns=["item", "status", "bukti"]).to_csv(DOCS / "data_ethics_checklist.csv", index=False, encoding="utf-8")

bps_path = DOCS / "bps_input.csv"
bps = pd.read_csv(bps_path) if bps_path.exists() else pd.DataFrame()
jumlah_osm = len(tabel)
if len(bps):
    pb = bps.copy()
    pb["jumlah_osm"] = jumlah_osm
    pb["selisih"] = pb["jumlah_bps"] - jumlah_osm
    pb["persen_osm_terhadap_bps"] = (jumlah_osm / pb["jumlah_bps"] * 100).round(1)
    pb["keterangan"] = "OSM dibandingkan angka BPS; definisi cakupan usaha dapat berbeda"
    for k in ("tabel", "halaman"):
        if k not in pb:
            pb[k] = None
else:
    pb = pd.DataFrame([{
        "sumber": "tidak ditemukan", "tahun": None, "jumlah_osm": jumlah_osm, "jumlah_bps": None,
        "selisih": None, "persen_osm_terhadap_bps": None, "tabel": None, "halaman": None,
        "keterangan": "Angka BPS per kecamatan untuk Kintamani tidak ditemukan pada sumber yang diperiksa",
    }])
pb = pb[["sumber", "tahun", "tabel", "halaman", "jumlah_osm", "jumlah_bps", "selisih", "persen_osm_terhadap_bps", "keterangan"]]
pb.to_csv(DOCS / "perbandingan_bps.csv", index=False, encoding="utf-8")

print(pd.DataFrame(kamus, columns=["kolom", "tipe", "deskripsi", "contoh", "boleh_kosong", "persen_terisi"]).to_string(index=False))
print(pd.DataFrame(etika, columns=["item", "status", "bukti"]).to_string(index=False))
print(pb.to_string(index=False))

       kolom           tipe                                             deskripsi                                        contoh boleh_kosong  persen_terisi
    osm_type           teks             Jenis objek OSM: node, way, atau relation                                          node        Tidak          100.0
      osm_id bilangan bulat                             ID objek di OpenStreetMap                                    1977779907        Tidak          100.0
        nama           teks                        Nama akomodasi sesuai tag name                       The Lakeview Hotel Bali           Ya           96.9
       jenis           teks                                     Nilai tag tourism                                         hotel        Tidak          100.0
         lat        desimal      Lintang titik atau titik tengah bangunan (WGS84)                                    -8.2831398        Tidak          100.0
         lon        desimal        Bujur titik atau titik tengah

In [ ]:
indeks = {(e["type"], e["id"]): e for e in data["elements"]}
sampel = tabel.sample(n=min(10, len(tabel)), random_state=int(NIM)).sort_values(["osm_type", "osm_id"]).reset_index(drop=True)


def cocok(baris):
    e = indeks.get((baris["osm_type"], baris["osm_id"]), {})
    tg = e.get("tags", {})
    nama_sama = (tg.get("name") or None) == (baris["nama"] if pd.notna(baris["nama"]) else None)
    return "ya" if nama_sama and tg.get("tourism") == baris["jenis"] else "tidak"


sampel["cocok_respons_mentah"] = sampel.apply(cocok, axis=1)
if poligon is not None:
    titik_sampel = gpd.GeoSeries(gpd.points_from_xy(sampel["lon"], sampel["lat"]), crs="EPSG:4326")
    sampel["di_dalam_poligon"] = titik_sampel.intersects(poligon).map({True: "ya", False: "tidak"}).values
else:
    sampel["di_dalam_poligon"] = "tidak dicek"
sampel["status_verifikasi"] = ((sampel["cocok_respons_mentah"] == "ya") & (sampel["di_dalam_poligon"] == "ya")).map({True: "terverifikasi otomatis", False: "perlu dicek"}).values
sampel["verifikasi_manual"] = "belum diperiksa"
sampel["catatan_manual"] = ""

vf_path = DOCS / "verifikasi_sampel.csv"
kunci = ["osm_type", "osm_id"]
if vf_path.exists():
    lama = pd.read_csv(vf_path)
    if set(kunci + ["verifikasi_manual", "catatan_manual"]) <= set(lama.columns):
        lama = lama.drop_duplicates(kunci).set_index(kunci)[["verifikasi_manual", "catatan_manual"]]
        gabung = sampel.join(lama, on=kunci, rsuffix="_lama")
        sampel["verifikasi_manual"] = gabung["verifikasi_manual_lama"].fillna(sampel["verifikasi_manual"])
        sampel["catatan_manual"] = gabung["catatan_manual_lama"].fillna(sampel["catatan_manual"])
    else:
        shutil.copy2(vf_path, DOCS / "verifikasi_sampel_sebelumnya.csv")

sampel = sampel[["osm_type", "osm_id", "nama", "jenis", "desa", "website", "url_osm", "cocok_respons_mentah", "di_dalam_poligon", "status_verifikasi", "verifikasi_manual", "catatan_manual"]]
sampel.to_csv(vf_path, index=False, encoding="utf-8")

print(sampel[["osm_type", "osm_id", "nama", "jenis", "cocok_respons_mentah", "di_dalam_poligon", "status_verifikasi", "verifikasi_manual"]].to_string(index=False))
print("Terverifikasi otomatis:", int((sampel["status_verifikasi"] == "terverifikasi otomatis").sum()), "dari", len(sampel))

osm_type      osm_id                              nama       jenis cocok_respons_mentah di_dalam_poligon      status_verifikasi verifikasi_manual
    node  2107226701                            Arlina guest_house                   ya               ya terverifikasi otomatis   belum diperiksa
    node  4297310696                    The Cave Hotel       hotel                   ya               ya terverifikasi otomatis   belum diperiksa
    node  4863479822     Miranda Homestay & Restaurant guest_house                   ya               ya terverifikasi otomatis   belum diperiksa
    node  5175562421 Entrance to: Bunbulan Hill Hostel      hostel                   ya               ya terverifikasi otomatis   belum diperiksa
    node  5668936157                 Black Lava Hostel      hostel                   ya               ya terverifikasi otomatis   belum diperiksa
    node  7205318685            Tegal Suci Guest House guest_house                   ya               ya terverifikasi otoma

In [ ]:
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.platypus import Image, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

REFLEKSI = (
    "Data akomodasi dari OpenStreetMap tidak boleh dianggap lengkap karena seluruh isinya dikontribusikan relawan "
    "secara sukarela. Tidak ada kewajiban bagi pemilik usaha maupun pemeta untuk memperbarui data, sehingga akomodasi "
    "baru, usaha kecil seperti homestay dan kamar sewa rumahan, serta usaha yang tidak dikenal pemeta sering tidak "
    "tercatat. Kelengkapan atribut juga tidak merata: banyak objek tidak memiliki nama, desa, bintang, jumlah kamar, "
    "maupun situs resmi, sehingga analisis yang bergantung pada atribut tersebut menjadi bias. Klasifikasi jenis, "
    "terutama villa, bergantung pada keputusan masing-masing pemeta karena tag tourism yang dipakai beragam. Hasil "
    "query hanya memotret satu waktu akses, sedangkan kondisi di lapangan terus berubah. Karena itu jumlah dari OSM "
    "sebaiknya dibaca sebagai batas bawah, bukan angka resmi. Untuk riset lanjutan, sumber yang perlu ditambahkan "
    "adalah publikasi resmi BPS Kabupaten Bangli dan dinas pariwisata, daftar usaha berizin dari perizinan daerah, "
    "serta survei lapangan atau konfirmasi langsung kepada pengelola secara etis. Platform pemesanan hanya dapat "
    "dipakai bila ada izin atau kerja sama resmi, bukan melalui scraping."
)
jumlah_kata = len(REFLEKSI.split())
assert 150 <= jumlah_kata <= 200, jumlah_kata

gaya = getSampleStyleSheet()
isi = ParagraphStyle("isi", parent=gaya["BodyText"], fontSize=9.5, leading=13)
sel = ParagraphStyle("sel", parent=isi, fontSize=8, leading=10)
sel_b = ParagraphStyle("sel_b", parent=sel, fontName="Helvetica-Bold")
sub = ParagraphStyle("sub", parent=gaya["Heading2"], fontSize=12, spaceBefore=8, spaceAfter=4)


def P(teks, g=isi):
    return Paragraph(escape(str(teks)), g)


def tabel_pdf(baris, lebar):
    isi_tabel = [[P(c, sel_b if i == 0 else sel) for c in b] for i, b in enumerate(baris)]
    t = Table(isi_tabel, colWidths=lebar, repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#E6E6E6")),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ]))
    return t


def kaki(canvas, doc):
    canvas.saveState()
    canvas.setFont("Helvetica", 7.5)
    canvas.drawString(2 * cm, 1.1 * cm, f"© OpenStreetMap contributors (ODbL) | {NAMA} - {NIM}")
    canvas.drawRightString(A4[0] - 2 * cm, 1.1 * cm, f"Halaman {doc.page}")
    canvas.restoreState()


metode_teks = {
    "area_administratif": "batas administratif Kecamatan Kintamani (admin_level 6)",
    "bounding_box": "kotak wilayah (bounding box) karena batas administratif tidak ditemukan",
}[prov["metode_wilayah"]]
filter_teks = f"Titik koordinat kemudian disaring terhadap poligon batas Kecamatan Kintamani (relasi OSM {RELATION_ID}, admin_level 6) dengan geopandas dan shapely." if poligon is not None else "Poligon batas tidak tersedia sehingga titik disaring dengan bounding box."
lebar_penuh = A4[0] - 4 * cm

cerita = [
    Paragraph("Laporan Praktik: Akuisisi Data Akomodasi di Kecamatan Kintamani", gaya["Title"]),
    P(f"Mata kuliah BIFP-243 Akuisisi dan Manajemen Data | {NAMA} | NIM {NIM}"),
    P(f"Sumber: OpenStreetMap melalui Overpass API, diakses {prov['waktu_akuisisi_utc']} (UTC). Lisensi ODbL. Atribusi: © OpenStreetMap contributors."),
    Paragraph("1. Ringkasan dataset", sub),
    P(f"Query mengambil {len(data['elements'])} objek dari respons mentah dengan cakupan {metode_teks}. {filter_teks} Setelah pemeriksaan dasar, dataset akhir memuat {len(tabel)} akomodasi. Jenis yang diambil adalah hotel, guest_house, hostel, motel, apartment, dan chalet. Villa di OSM umumnya diberi tag guest_house, chalet, atau apartment, sehingga ketiganya dimasukkan agar villa tercakup; konsekuensinya sebagian guest_house dan apartment non-villa ikut terhitung."),
    Spacer(1, 6),
    tabel_pdf([["Jenis", "Jumlah"]] + per_jenis.values.tolist(), [6 * cm, 3 * cm]),
    Spacer(1, 6),
]
desa_atas = per_desa.head(12)
sisa = int(per_desa["jumlah"].iloc[12:].sum())
baris_desa = [["Desa", "Jumlah"]] + desa_atas.values.tolist()
if sisa:
    baris_desa.append([f"Desa lainnya ({len(per_desa) - 12} desa)", sisa])
cerita += [P(f"Sebaran per desa (12 teratas dari {len(per_desa)} kategori; daftar lengkap di docs/ringkasan_desa.csv):"), Spacer(1, 4), tabel_pdf(baris_desa, [8 * cm, 3 * cm])]

cerita += [
    Paragraph("2. Peta sebaran", sub),
    Image(str(REPORTS / "peta_sebaran.png"), width=11.5 * cm, height=11.5 * cm * 6 / 7),
    P("Peta berupa plot titik koordinat (lat, lon) dari dataset akhir. Data peta © OpenStreetMap contributors."),
    Paragraph("3. Uji kelengkapan terhadap BPS dan verifikasi sampel", sub),
]
if pb["jumlah_bps"].notna().any():
    for _, r in pb.iterrows():
        rujukan = ", ".join(str(x) for x in (r["sumber"], r["tahun"], r["tabel"], r["halaman"]) if pd.notna(x))
        cerita.append(P(f"{rujukan}: BPS mencatat {int(r['jumlah_bps'])} akomodasi, OSM {int(r['jumlah_osm'])} atau {r['persen_osm_terhadap_bps']}% dari angka BPS (selisih {int(r['selisih'])}). Definisi usaha akomodasi BPS dan tag OSM tidak sepenuhnya sama, sehingga perbandingan ini bersifat indikatif."))
else:
    cerita.append(P("Angka akomodasi per kecamatan untuk Kintamani tidak ditemukan pada publikasi dan portal data yang diperiksa (Bangli Dalam Angka, Kecamatan Kintamani Dalam Angka, Satu Data Bali). Sesuai ketentuan tugas, hal ini dijelaskan di sini dan perbandingan numerik terhadap BPS belum dapat dihitung. Jumlah OSM hasil akuisisi adalah " + str(jumlah_osm) + " akomodasi."))
cerita += [Spacer(1, 6), P(f"Verifikasi {len(sampel)} sampel acak (seed = NIM). Status 'terverifikasi otomatis' berarti nama dan jenis cocok dengan respons mentah dan titik berada di dalam poligon batas; kolom verifikasi manual mencatat pengecekan nama, jenis, dan lokasi lewat tautan OSM atau situs resmi akomodasi.")]
baris_sampel = [["OSM", "Nama", "Jenis", "Status otomatis", "Verifikasi manual"]]
for _, r in sampel.iterrows():
    baris_sampel.append([f"{r['osm_type']}/{r['osm_id']}", r["nama"] if pd.notna(r["nama"]) else "(tanpa nama)", r["jenis"], r["status_verifikasi"], r["verifikasi_manual"]])
cerita += [tabel_pdf(baris_sampel, [3.0 * cm, 5.2 * cm, 2.4 * cm, 3.4 * cm, 3.0 * cm])]

cerita += [Paragraph("4. Masalah kualitas data dan keputusan", sub), tabel_pdf([["Pemeriksaan", "Temuan", "Keputusan"]] + [[a, b, c] for a, b, c in catatan], [6 * cm, 2.2 * cm, 8.8 * cm])]
gagal = [x for x in prov["log_percobaan"] if "status 200" not in x or "runtime" in x]
if gagal:
    kendala = "Server Overpass sempat bermasalah selama akuisisi (" + "; ".join(prov["log_percobaan"]) + "). Permintaan diulang dengan jeda bertahap sesuai fungsi retry."
else:
    kendala = "Tidak ada kendala server; permintaan berhasil dengan status 200 (" + "; ".join(prov["log_percobaan"]) + ")."
cerita += [Spacer(1, 6), P("Kendala server Overpass: " + kendala)]
cerita += [Paragraph("5. Refleksi", sub), P(REFLEKSI)]

doc = SimpleDocTemplate(str(REPORTS / "laporan_akomodasi_kintamani.pdf"), pagesize=A4, leftMargin=2 * cm, rightMargin=2 * cm, topMargin=1.8 * cm, bottomMargin=1.8 * cm, title="Laporan Akuisisi Data Akomodasi Kintamani", author=NAMA)
doc.build(cerita, onFirstPage=kaki, onLaterPages=kaki)
HALAMAN = doc.page

print("Jumlah kata refleksi:", jumlah_kata)
print("Jumlah halaman laporan:", HALAMAN)
print("Laporan tersimpan:", REPORTS / "laporan_akomodasi_kintamani.pdf")

Jumlah kata refleksi: 160
Jumlah halaman laporan: 3
Laporan tersimpan: /content/reports/laporan_akomodasi_kintamani.pdf


In [ ]:
readme = f"""# Akuisisi Data Akomodasi di Kecamatan Kintamani

Tugas praktik BIFP-243 Akuisisi dan Manajemen Data, INSTIKI.
Pengambil: {NAMA} (NIM {NIM})

## Tujuan

Mengakuisisi data akomodasi di Kecamatan Kintamani dari API publik secara etis, menyimpan respons mentah, mengubahnya menjadi dataset tabular terdokumentasi, dan menilai kelengkapannya terhadap sumber resmi.

## Sumber data

- OpenStreetMap melalui Overpass API ({prov['url_endpoint']})
- Pembanding: publikasi resmi BPS Kabupaten Bangli (lihat docs/perbandingan_bps.csv)

Atribusi: © OpenStreetMap contributors. Data dilisensikan di bawah Open Database License (ODbL) 1.0: https://www.openstreetmap.org/copyright

## Tanggal akses dan metode akuisisi

- Waktu akuisisi (UTC): {prov['waktu_akuisisi_utc']}
- Metode wilayah: {prov['metode_wilayah']}
- Jumlah elemen respons mentah: {prov['jumlah_elemen']}
- Query: docs/query_overpass.txt
- Respons mentah: data/raw/{prov['berkas_mentah']['nama']} (MD5 {prov['berkas_mentah']['md5']}), tidak diubah setelah disimpan
- Batas wilayah: relasi OSM {RELATION_ID}{(' (data/raw/' + prov['berkas_batas']['nama'] + ')') if prov.get('berkas_batas') else ' tidak berhasil diambil; memakai bounding box'}
- Dataset tabular: data/interim/akomodasi_kintamani.csv ({len(tabel)} baris)

## Batasan etika

- Hanya memakai Overpass API; tidak ada scraping OTA, platform pemesanan, atau Google Maps.
- Data kontak (telepon, email, akun media sosial pemilik) tidak diambil dan tidak dicantumkan.
- Permintaan ke server dibatasi dan memakai User-Agent yang teridentifikasi.
- Data OSM diisi relawan sehingga tidak lengkap; jumlah dibaca sebagai batas bawah.
- Rincian: docs/data_ethics_checklist.csv

## Cara menjalankan ulang

1. Pasang dependensi: pip install pandas requests matplotlib reportlab geopandas shapely
2. Buka notebooks/akuisisi_akomodasi.ipynb dari folder notebooks atau dari akar repositori.
3. Jalankan semua sel dari awal. Bila data/raw dan docs/provenance.json sudah ada dan hash cocok, respons mentah dipakai ulang tanpa request baru.
4. Untuk akuisisi baru, setel variabel lingkungan FORCE_FETCH=1 sebelum menjalankan.
5. Untuk perbandingan BPS, buat docs/bps_input.csv berkolom sumber,tahun,jumlah_bps lalu jalankan ulang.

## Struktur keluaran

- data/raw: respons mentah Overpass
- data/interim: dataset tabular
- docs: query, provenance, data dictionary, data ethics checklist, perbandingan BPS, verifikasi sampel, catatan kualitas
- reports: laporan PDF dan peta sebaran
"""
(ROOT / "README.md").write_text(readme, encoding="utf-8")
print(readme[:600])
print("README tersimpan:", ROOT / "README.md")

# Akuisisi Data Akomodasi di Kecamatan Kintamani

Tugas praktik BIFP-243 Akuisisi dan Manajemen Data, INSTIKI.
Pengambil: I Gede Raditya Ananda Putra (NIM 2501010024)

## Tujuan

Mengakuisisi data akomodasi di Kecamatan Kintamani dari API publik secara etis, menyimpan respons mentah, mengubahnya menjadi dataset tabular terdokumentasi, dan menilai kelengkapannya terhadap sumber resmi.

## Sumber data

- OpenStreetMap melalui Overpass API (https://overpass-api.de/api/interpreter)
- Pembanding: publikasi resmi BPS Kabupaten Bangli (lihat docs/perbandingan_bps.csv)

Atribusi: © OpenStreetMap contr
README tersimpan: /content/README.md


In [ ]:
wajib = [
    RAW / prov["berkas_mentah"]["nama"],
    INTERIM / "akomodasi_kintamani.csv",
    DOCS / "query_overpass.txt",
    DOCS / "provenance.json",
    DOCS / "data_dictionary.csv",
    DOCS / "data_ethics_checklist.csv",
    DOCS / "perbandingan_bps.csv",
    REPORTS / "laporan_akomodasi_kintamani.pdf",
    ROOT / "README.md",
]
hasil = []
hasil.append(("semua berkas keluaran ada", all(p.exists() for p in wajib)))
hasil.append(("kolom dataset sesuai struktur bagian 6", list(pd.read_csv(INTERIM / "akomodasi_kintamani.csv").columns) == KOLOM))
hasil.append(("respons mentah utuh (MD5 cocok)", md5_berkas(RAW / prov["berkas_mentah"]["nama"]) == prov["berkas_mentah"]["md5"]))
hasil.append(("tanpa duplikat osm_type + osm_id", not pd.read_csv(INTERIM / "akomodasi_kintamani.csv").duplicated(["osm_type", "osm_id"]).any()))
hasil.append(("tanpa kolom atau nilai kontak pribadi", tanpa_kolom_kontak and tanpa_nilai_kontak))
print("Filter wilayah:", metode_filter)
hasil.append(("atribusi OSM ada di README", "© OpenStreetMap contributors" in (ROOT / "README.md").read_text(encoding="utf-8")))
hasil.append(("laporan 3 sampai 5 halaman", 3 <= HALAMAN <= 5))
hasil.append(("refleksi 150 sampai 200 kata", 150 <= jumlah_kata <= 200))
hasil.append(("query final tersimpan", (DOCS / "query_overpass.txt").stat().st_size > 0))

for nama_cek, ok in hasil:
    print("[OK]" if ok else "[GAGAL]", nama_cek)
assert all(ok for _, ok in hasil)
print("Semua pemeriksaan otomatis lulus.")

Filter wilayah: poligon relasi OSM 20447233
[OK] semua berkas keluaran ada
[OK] kolom dataset sesuai struktur bagian 6
[OK] respons mentah utuh (MD5 cocok)
[OK] tanpa duplikat osm_type + osm_id
[OK] tanpa kolom atau nilai kontak pribadi
[OK] atribusi OSM ada di README
[OK] laporan 3 sampai 5 halaman
[OK] refleksi 150 sampai 200 kata
[OK] query final tersimpan
Semua pemeriksaan otomatis lulus.


In [ ]:
import zipfile
from google.colab import files

NAMA_FOLDER = f"akomodasi_kintamani_{NIM}"
with zipfile.ZipFile(f"{NAMA_FOLDER}.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for sub in ("data", "docs", "reports"):
        for p in (ROOT / sub).rglob("*"):
            if p.is_file():
                z.write(p, Path(NAMA_FOLDER) / p.relative_to(ROOT))
    z.write(ROOT / "README.md", Path(NAMA_FOLDER) / "README.md")
files.download(f"{NAMA_FOLDER}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>